# Teks — Embedding banyak model (Qwen3-Embedding, bge-m3, e5) + kepala klasik + stacking
**Kapan dipakai:** klasifikasi teks, ingin skor kuat TANPA fine-tune (cepat, aman VRAM), atau
ingin fitur beragam untuk `ensemble_oof`. Tiap model embedding dihitung sekali (disimpan di cache
`.npy`), lalu kepala ringan (LogisticRegression / LightGBM) dilatih k-fold di atasnya.

**Beda dengan `klasifikasi_embedding`:** di sini banyak model sekaligus, pooling sesuai jenis
model (`last` untuk model decoder seperti Qwen3-Embedding, `cls` untuk bge-m3, `mean` untuk
e5/MiniLM), awalan instruksi per model, OOF per model + OOF gabungan (embedding disambung).

**Alur:** Konfigurasi → baca → split → embedding tiap model (cache) → kepala k-fold per model →
gabungan semua embedding → pilih terbaik / rata-rata → submission → validator.

**Saat lomba yang diubah:** sel Konfigurasi (`# GANTI saat lomba`) dan isi `EMBEDDERS`. Model yang
foldernya belum ada di `pretrained/` dilewati otomatis (unduh dengan `download_model.bat`).

**Data demo:** SmSA (`sentiment`, 3 kelas).

In [ ]:
import html
import os
import time

import numpy as np
import pandas as pd
import torch
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.nn.functional import normalize
from transformers import AutoModel, AutoTokenizer

## Konfigurasi
`EMBEDDERS`: nama → folder, pooling, awalan, panjang maks. Awalan instruksi penting:
Qwen3-Embedding dan e5-instruct dilatih dengan `Instruct: ...` lalu `Query: `, e5 biasa dengan
`query: `, bge-m3 tanpa awalan.

In [ ]:
DATA_FOLDER = "../../../data_latihan/text/sentiment"          # GANTI saat lomba
TRAIN_CSV = f"{DATA_FOLDER}/smsa_train.csv"                    # GANTI saat lomba
TEST_CSV = f"{DATA_FOLDER}/smsa_test.csv"                      # GANTI saat lomba
OUTPUT_FOLDER = "../../../outputs/text/embedding_llm_oof"
OOF_FOLDER = "../../../outputs/text/oof"
CACHE_FOLDER = f"{OUTPUT_FOLDER}/cache"   # embedding .npy, hapus bila data/awalan berubah
SAMPLE_CSV = f"{OUTPUT_FOLDER}/sample_submission_demo.csv"     # GANTI saat lomba: file panitia
SUBMISSION_NAME = "submission.csv"
ANSWER_CSV = f"{DATA_FOLDER}/smsa_test_labels.csv"             # HAPUS saat lomba (kunci demo)
ID_COL = "id"                    # GANTI saat lomba
TEXT_COL = "text"                # GANTI saat lomba
TARGET = "label"                 # GANTI saat lomba
SUBMIT_COL = TARGET              # GANTI saat lomba bila kolom prediksi di sample bernama lain
N_FOLDS = 5
SPLIT_SEED = 42                  # tetap 42 agar OOF sefold dengan notebook lain
NOTEBOOK = "embedding_llm_oof"
TASK = "Classify the sentiment of the given Indonesian text"   # GANTI: tugas dalam 1 kalimat
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = DEVICE == "cuda"
EMBED_BATCH = 16                 # turunkan ke 8 bila VRAM habis (model besar, MAX panjang)

In [ ]:
PRE = "../../../pretrained"
INSTRUCT = f"Instruct: {TASK}\nQuery: "
EMBEDDERS = {
    "qwen3_emb": {"path": f"{PRE}/qwen3_emb_06b", "pool": "last", "prefix": INSTRUCT, "max": 512},
    "e5_base": {"path": f"{PRE}/e5_base", "pool": "mean", "prefix": "query: ", "max": 512},
    "minilm": {"path": f"{PRE}/minilm", "pool": "mean", "prefix": "", "max": 256},
}
# Tambahan (hapus # untuk ikut dihitung; folder dari download_model.bat):
# EMBEDDERS["bge_m3"] = {"path": f"{PRE}/bge_m3", "pool": "cls", "prefix": "", "max": 512}
# EMBEDDERS["e5_large_inst"] = {"path": f"{PRE}/e5_large_instruct", "pool": "mean",
#                               "prefix": INSTRUCT, "max": 512}   # e5 besar + instruksi, kuat
# EMBEDDERS["indobert_mean"] = {"path": f"{PRE}/indobert", "pool": "mean", "prefix": "",
#                               "max": 512}   # BERT biasa tanpa fine-tune: lemah, untuk ragam saja
HEAD_C = 2.0                     # kekuatan regularisasi LogisticRegression (besar = longgar)
USE_CONCAT = True                # tambah satu kepala di atas semua embedding yang disambung


def make_head():
    return LogisticRegression(C=HEAD_C, max_iter=3000)
    # return LogisticRegression(C=HEAD_C, max_iter=3000, class_weight="balanced")  # kelas timpang
    # return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000))
    # return LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=31, verbose=-1)


print("embedder:", list(EMBEDDERS), "| device:", DEVICE)

## Metrik
Ganti baris `return` bila metrik panitia lain. Semua di dalam memakai indeks kelas 0..K-1.

In [ ]:
METRIC_NAME = "f1_macro"         # GANTI saat lomba, ikut baris return di bawah


def score(truth, predicted):
    return f1_score(truth, predicted, average="macro")
    # return f1_score(truth, predicted, average="weighted")


print("metrik:", METRIC_NAME)

## Baca data
Teks kosong/NaN jadi string kosong, lalu `clean_basic` (html + spasi rapi). Label diubah ke indeks
0..K-1 urut `np.unique` (format OOF bersama).

In [ ]:
os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(OOF_FOLDER, exist_ok=True)
train = pd.read_csv(TRAIN_CSV)
test = pd.read_csv(TEST_CSV)


def clean_basic(text):
    return " ".join(html.unescape(text).split())


train_text = train[TEXT_COL].fillna("").astype(str)
test_text = test[TEXT_COL].fillna("").astype(str)
train_text = train_text.apply(clean_basic)
test_text = test_text.apply(clean_basic)
classes = np.unique(train[TARGET])
y = np.searchsorted(classes, train[TARGET].to_numpy())
print("train:", train.shape, "| test:", test.shape, "| NaN target:", train[TARGET].isna().sum())
print("kelas:", list(classes), "| jumlah:", np.bincount(y).tolist())
train[[ID_COL, TEXT_COL, TARGET]].head()

## Buat sample_submission demo (HAPUS saat lomba)
Data latihan tidak punya `sample_submission`. Saat lomba file panitia dipakai (`SAMPLE_CSV`);
sel ini tidak menimpa file di luar `OUTPUT_FOLDER`.

In [ ]:
demo_sample = pd.DataFrame()
demo_sample[ID_COL] = test[ID_COL]
demo_sample[SUBMIT_COL] = train[TARGET].iloc[0]
if SAMPLE_CSV.startswith(OUTPUT_FOLDER):
    demo_sample.to_csv(SAMPLE_CSV, index=False)
    print("sample demo ditulis ke", SAMPLE_CSV)
else:
    print("SAMPLE_CSV bukan file demo: dibiarkan apa adanya")

In [ ]:
sample = pd.read_csv(SAMPLE_CSV)
print("sample_submission:", sample.shape, sample.columns.tolist())
print("id test = id sample:", set(test[ID_COL]) == set(sample[ID_COL]))
sample.head(3)

## Split (k-fold)
Aturan format OOF: `StratifiedKFold` acak seed `SPLIT_SEED` pada urutan baris train apa adanya,
jadi OOF notebook ini sefold dengan `finetune_oof`, `baseline_oof`, dll. dan bisa di-ensemble.

In [ ]:
splitter = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SPLIT_SEED)
folds = list(splitter.split(train_text, y))
val_sizes = []
for _, val_index in folds:
    val_sizes.append(len(val_index))
print("fold:", len(folds), "| ukuran validasi:", val_sizes)

## Hitung embedding (sekali per model, lalu dari cache)
`pool`: `last` = vektor token terakhir yang bukan padding (model decoder: Qwen3-Embedding, LLM),
`cls` = token pertama (bge-m3), `mean` = rata-rata token (e5, MiniLM, BERT). Vektor dinormalkan
L2. Teks diurut panjangnya agar padding minimal.

In [ ]:
def pool(hidden, mask, how):
    if how == "cls":
        return hidden[:, 0]
    if how == "last":
        last = mask.sum(dim=1) - 1          # tokenizer padding kanan: token asli terakhir
        rows = torch.arange(hidden.shape[0], device=hidden.device)
        return hidden[rows, last]
    weights = mask.unsqueeze(-1).to(hidden.dtype)
    return (hidden * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1)


def load_embedder(spec):
    tokenizer = AutoTokenizer.from_pretrained(spec["path"], local_files_only=True)
    tokenizer.padding_side = "right"
    model = AutoModel.from_pretrained(spec["path"], local_files_only=True)
    model = model.to(DEVICE)
    return tokenizer, model.eval()


print("pool / load_embedder siap")

In [ ]:
def embed_batch(tokenizer, model, spec, batch_texts):
    batch = tokenizer(batch_texts, padding=True, truncation=True, max_length=spec["max"],
                      return_tensors="pt").to(DEVICE)
    with torch.autocast("cuda", dtype=torch.bfloat16, enabled=USE_BF16):
        hidden = model(**batch).last_hidden_state
    vectors = normalize(pool(hidden.float(), batch["attention_mask"], spec["pool"]), dim=1)
    return vectors.cpu().numpy()


def embed_texts(spec, texts):
    tokenizer, model = load_embedder(spec)
    prefixed = []
    for text in texts:
        prefixed.append(spec["prefix"] + text)
    order = np.argsort(texts.str.len().to_numpy())
    vectors = np.zeros((len(texts), model.config.hidden_size), dtype="float32")
    with torch.no_grad():
        for start in range(0, len(order), EMBED_BATCH):
            chosen = order[start:start + EMBED_BATCH]
            batch_texts = []
            for number in chosen:
                batch_texts.append(prefixed[number])
            vectors[chosen] = embed_batch(tokenizer, model, spec, batch_texts)
    del model
    torch.cuda.empty_cache()
    return vectors


print("embed_texts siap")

In [ ]:
os.makedirs(CACHE_FOLDER, exist_ok=True)
all_text = pd.concat([train_text, test_text], ignore_index=True)
features = {}
embed_seconds = {}
for name, spec in EMBEDDERS.items():
    cache = f"{CACHE_FOLDER}/{name}.npy"
    if not os.path.isdir(spec["path"]):
        print(f"{name}: folder {spec['path']} tidak ada, dilewati (download_model.bat)")
        continue
    began = time.time()
    if os.path.exists(cache):
        vectors = np.load(cache)
    else:
        vectors = embed_texts(spec, all_text)
        np.save(cache, vectors)
    features[name] = vectors
    embed_seconds[name] = round(time.time() - began, 1)
    print(f"{name}: {vectors.shape} | {embed_seconds[name]} dtk")
print("embedding siap:", list(features))

## Kepala k-fold per model
Satu kepala per fold di atas embedding; OOF = prediksi validasi, test = rata-rata 5 kepala fold.
`USE_CONCAT`: semua embedding disambung jadi satu matriks (kombinasi sering menang tipis).

In [ ]:
def run_head(matrix):
    train_x = matrix[:len(train)]
    test_x = matrix[len(train):]
    oof = np.zeros((len(train), len(classes)), dtype="float32")
    test_proba = np.zeros((len(test), len(classes)), dtype="float32")
    for fit_index, val_index in folds:
        head = make_head()
        head.fit(train_x[fit_index], y[fit_index])
        oof[val_index] = head.predict_proba(train_x[val_index])
        test_proba += head.predict_proba(test_x) / len(folds)
    return oof, test_proba


if USE_CONCAT and len(features) > 1:
    features["concat"] = np.hstack(list(features.values()))
print("kepala untuk:", list(features))

In [ ]:
results = {}
log_path = f"{OOF_FOLDER}/percobaan.csv"
for name, matrix in features.items():
    began = time.time()
    oof, test_proba = run_head(matrix)
    value = score(y, oof.argmax(axis=1))
    results[name] = {"oof": oof, "test": test_proba, "score": value}
    np.save(f"{OOF_FOLDER}/oof_emb_{name}.npy", oof)
    np.save(f"{OOF_FOLDER}/test_emb_{name}.npy", test_proba)
    seconds = round(time.time() - began + embed_seconds.get(name, 0), 1)
    row = {"waktu": time.strftime("%Y-%m-%d %H:%M:%S"), "notebook": NOTEBOOK,
           "model": f"emb_{name}", "metrik": METRIC_NAME, "skor_cv": round(value, 4),
           "detik": seconds, "catatan": f"kepala {type(make_head()).__name__} C {HEAD_C}"}
    pd.DataFrame([row]).to_csv(log_path, mode="a", header=not os.path.exists(log_path),
                               index=False)
    print(f"{name}: {METRIC_NAME} OOF {value:.4f} | {seconds} dtk")

## Pilih hasil akhir
`best` = model dengan OOF tertinggi. Alternatif: rata-rata probabilitas semua model (sering lebih
stabil di test). Ensemble lanjutan (bobot, hill climbing, stacking) ada di `ensemble_oof`.

In [ ]:
ranking = pd.Series(dtype=float)
for name, item in results.items():
    ranking[name] = item["score"]
ranking = ranking.sort_values(ascending=False)
print(ranking.round(4).to_string())
best = ranking.index[0]
oof = results[best]["oof"]
test_proba = results[best]["test"]
# rata-rata semua model (hapus # pada 4 baris di bawah):
# oof = np.mean([item["oof"] for item in results.values()], axis=0)
# test_proba = np.mean([item["test"] for item in results.values()], axis=0)
# best = "rata_rata"
oof_score = score(y, oof.argmax(axis=1))
print(f"dipakai: {best} | {METRIC_NAME} OOF {oof_score:.4f}")
print(classification_report(y, oof.argmax(axis=1), target_names=classes.astype(str)))

## Submission
Kolom id diambil dari `sample_submission`; prediksi dipetakan lewat id.

In [ ]:
test_pred = test_proba.argmax(axis=1)
pred_by_id = pd.Series(classes[test_pred], index=test[ID_COL])

submission = pd.DataFrame()
submission[ID_COL] = sample[ID_COL]
submission[SUBMIT_COL] = sample[ID_COL].map(pred_by_id)
submission.to_csv(f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}", index=False)
predicted_share = submission[SUBMIT_COL].value_counts(normalize=True)
print("sebaran prediksi test:", predicted_share.round(3).to_dict())
submission.head()

## Cek submission terhadap sample_submission
Salinan validator `arsenal/umum/simpan_muat` #15 dan #16: hanya membaca dan mencetak. Hasil
akhir harus `MASALAH: 0`.

In [ ]:
SUBMISSION_CSV = f"{OUTPUT_FOLDER}/{SUBMISSION_NAME}"
ID_COLUMN = ID_COL

sample = pd.read_csv(SAMPLE_CSV)
written = pd.read_csv(SUBMISSION_CSV)
problems = []


def report(failed, text):
    if failed:
        problems.append(text)


wanted = sample[ID_COLUMN]
ids = written.get(ID_COLUMN, pd.Series(dtype=wanted.dtype))
lost = set(wanted) - set(ids)
extra = set(ids) - set(wanted)
kept = ids[ids.isin(wanted)].drop_duplicates().tolist()
expected = wanted[wanted.isin(ids)].tolist()
report(written.columns.tolist() != sample.columns.tolist(), "kolom / urutan kolom beda")
report(len(written) != len(sample), f"jumlah baris {len(written)}, sample {len(sample)}")
report(lost, f"id hilang {len(lost)}, mis. {sorted(lost)[:3]}")
report(extra, f"id asing {len(extra)}, mis. {sorted(extra)[:3]}")
report(ids.duplicated().any(), f"id kembar: {ids.duplicated().sum()} baris")
report(kept != expected, "urutan id beda dari sample")

In [ ]:
LABEL_COLUMNS = [SUBMIT_COL]     # kolom isi kelas ([] = lewati cek label)
ALLOWED = list(classes)          # label sah = label train, [] = lewati cek

for column in sample.columns:
    if column in written.columns:
        empty = written[column].isna().sum()
        report(empty, f"{column}: {empty} nilai kosong")
        report(written[column].dtype != sample[column].dtype,
               f"{column}: tipe {written[column].dtype}, sample {sample[column].dtype}")
for column in LABEL_COLUMNS:
    if ALLOWED and column in written.columns:
        foreign = set(written[column].dropna()) - set(ALLOWED)
        report(foreign, f"{column}: label di luar ALLOWED {sorted(foreign)[:5]}")
        print(written[column].value_counts())   # semua satu kelas -> curigai model
for problem in problems:
    print("-", problem)
print("MASALAH:", len(problems))

## Cek dengan kunci jawaban (HAPUS saat lomba)
Hanya untuk data latihan yang punya kunci (`ANSWER_CSV`). Hapus sel ini saat lomba.

In [ ]:
answers = pd.read_csv(ANSWER_CSV)
truth_by_id = pd.Series(answers[TARGET].to_numpy(), index=answers[ID_COL])
truth_index = np.searchsorted(classes, truth_by_id.reindex(submission[ID_COL]).to_numpy())
pred_index = np.searchsorted(classes, submission[SUBMIT_COL].to_numpy())
print(f"{METRIC_NAME} di test (kunci): {score(truth_index, pred_index):.4f}")
print(f"{METRIC_NAME} OOF train: {oof_score:.4f}")